In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import *
from libs.dashcyto_lib import DASH_CYTO


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(


In [3]:
root0 = Path(dic_yml['root0'])
root_data = Path(dic_yml['root_data'])
root_colab = Path(dic_yml['root_colab'])

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

disease = dic_yml['disease']
gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

root_project = create_dir(root_data, s_project)
cfg = Config(root0=root0, root_project=root_project, project=project, s_project=s_project, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1
abs_lfc_cutoff, fdr_lfc_cutoff, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={abs_lfc_cutoff:.3f}; fdr={fdr_lfc_cutoff:.3f}  -  LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/target_disc_perturb_project/config/all_lfc_cutoffs_target_disc_perturb_project.tsv
project 'Perturbation agent: a molecular target discovery pipeline', s_project 'target_disc_perturb_project'
G/P LFC cutoffs: lfc=1.000; fdr=0.050  -  LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, LFC_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
mtd.echo_parameters()

Start opening tables ....
Building synonym dictionary ...

>>> Tumor


TypeError: Config.set_default_best_lfc_cutoff() got an unexpected keyword argument 'abs_lfc_cutoff'

In [ ]:
mtd.fname_given_lfc_table0, mtd.root_data

In [ ]:
mtd.set_lfc_names()

In [ ]:
mtd.fname_final_lfc_ori, mtd.root_data, mtd.root_result

In [ ]:
mtd.fname_given_lfc_table0, mtd.fname_final_lfc_table0

In [ ]:
mtd.case, mtd.group, mtd.gender, mtd.age, mtd.s_omics

In [ ]:
mtd.geneset_num, mtd.geneset_lib

In [ ]:
mtd.gene.df_my_gene.head(2)

### Check main cols

In [ ]:
mtd.check_lfc_names(verbose=True)

### Second case

In [ ]:
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
print("\nEcho Parameters:")
mtd.echo_parameters()

In [ ]:
print(case)
mtd.case, mtd.group, mtd.gender, mtd.age, mtd.s_omics, mtd.split_case(case)

In [ ]:
mtd.fname_final_lfc_ori, mtd.root_data, mtd.root_result

In [ ]:
fname, fname_cutoff = mtd.set_enrichment_name()
fname, os.path.exists(os.path.join(mtd.root_enrich, fname)), fname_cutoff, os.path.exists(os.path.join(mtd.root_enrich, fname_cutoff))

In [ ]:
try:
    dflfc_ori = mtd.dflfc_ori
    print(len(dflfc_ori))
except:
    dflfc_ori = pd.DataFrame()
    
dflfc_ori.head(3)

### Removing or Renaming config files only the defautl cutoffs are defined

In [ ]:
for case in case_list:
    print(">>>", case)
    ret, degs, fname_final_ori, dfdegs = mtd.open_case(case, verbose=False)

    if not ret: continue
    
    fname_final_ori, fname_ori, title = mtd.set_lfc_names()
    print(f"fname '{fname_final_ori}' and title '{title}'")
    print(f"LFC cutoff: lfc={mtd.abs_lfc_cutoff:.3f} fdr={mtd.fdr_lfc_cutoff}")
    
    print(f"{mtd.s_deg_dap}s = {len(degs)}\n")

In [ ]:
print(case)
mtd.split_case(case)
mtd.case, mtd.gender, mtd.age

In [ ]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(f"There are {len(dflfc_lnc)} LNCs\n")
dflfc_lnc.tail(3)

In [ ]:
dflfc_ori = mtd.dflfc_ori
print(len(dflfc_ori))
dflfc_ori_symb = dflfc_ori[~pd.isnull(dflfc_ori.symbol)]
print(len(dflfc_ori_symb))

### Microarray with 28,232 unique symbols

In [ ]:
try:
    symbols = np.unique(dflfc_ori.symbol)
except:
    symbols = []
    
len(symbols)

In [ ]:
try:
    dflfc = mtd.dflfc
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()
    
dflfc.head(3)

In [ ]:
dfbest = mtd.cfg.open_best_ptw_cutoff()
dfbest

In [ ]:
want_see_best_cutoff = False

if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
else:
    dfbest = pd.DataFrame()
dfbest    

In [ ]:
if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
    dfa = dfbest[(dfbest.case == case) & (dfbest.normalization == mtd.normalization) & (dfbest.geneset_num == geneset_num) ]
else:
    dfa = pd.DataFrame()

dfa

### Deleting or Renaming config files --> the default cutoffs are defined

In [ ]:
try:
    dflfc = mtd.dflfc_ori[(mtd.dflfc_ori.fdr < mtd.fdr_lfc_cutoff)]
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()

dflfc.head(3)

In [ ]:
for case in case_list:
    print(">>>", case)
    ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, verbose=False)

    if not ret: continue
    
    fname_final_ori, fname_ori, title = mtd.set_lfc_names()
    print(f"fname '{fname_final_ori}' and title '{title}'")
    print(f"LFC cutoff: lfc={mtd.abs_lfc_cutoff:.3f} fdr={mtd.fdr_lfc_cutoff}")
    
    print(f"{mtd.s_deg_dap}s = {len(degs)}\n")

### Minimum LFC cutoff

In [ ]:
np.log2(1.4)

### DEGs simulation: no DEG/DAPs per cases
### Saving simulation file dfsim in config:
  - all_lfc_cutoffs_taubate_covid19.tsv

#### Sampling

### Cutoff sets to generate the statistical data
  - inf lfc cutoff: 0.40 --> 0.48 ~ 40% modulation  --> 0.65
  - sup fdr cutoff: 0.75 --> no more than

In [ ]:
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
mtd.lfc_list = lfc_list
lfc_list[-1] = 0.0
lfc_list

In [ ]:
fdr_list = np.arange(0.05, 0.76, .01)
mtd.fdr_list = fdr_list
fdr_list

In [ ]:
cutoff_list = np.round([(x, y) for x in lfc_list for y in fdr_list],3)
print(len(cutoff_list))
cutoff_list[:5], cutoff_list[-5:]

### Saving simulations: calc_degs_cutoff_simulation()

  - config/all_lfc_cutoffs_medulloblastoma.tsv
  - while looping in case_list -> save_file -> save txt files

In [ ]:
cutoff_list

In [ ]:
verbose=False
save_file=False
force=True

# save_file
# in list_of_degs_set_params ... save excel files

dfsim = mtd.calc_degs_cutoff_simulation(cutoff_list=cutoff_list, force=force, save_file=force, n_echo=-1, verbose=verbose)
dfsim = dfsim.sort_values(['case', 'fdr_lfc_cutoff', 'abs_lfc_cutoff'], ascending=[False, True, False])
print(dfsim.columns)
print(len(dfsim))

In [ ]:
dfsim.head(2)

In [ ]:
dfsim.tail(2)

### Does the simulation worked?

In [ ]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))

dfsim2 = dfsim[dfsim.case == case]
dfsim2.head(3).T

In [ ]:
abs_lfc_cutoff = 0.95
fdr_lfc_cutoff = 0.05

# (dfsim.case == case) &
dfsim[ (dfsim.abs_lfc_cutoff == abs_lfc_cutoff) & (dfsim.fdr_lfc_cutoff == fdr_lfc_cutoff)].T

In [ ]:
np.unique(dfsim.abs_lfc_cutoff)

In [ ]:
dfsim.abs_lfc_cutoff.min(), dfsim.abs_lfc_cutoff.max(), 

In [ ]:
np.unique(dfsim.fdr_lfc_cutoff)

In [ ]:
dfsim.fdr_lfc_cutoff.min(), dfsim.fdr_lfc_cutoff.max(), 

In [ ]:
dfsim.abs_lfc_cutoff.min(), dfsim.abs_lfc_cutoff.max(), 

In [ ]:
dfsim.fdr_lfc_cutoff.min(), dfsim.fdr_lfc_cutoff.max(), 

### Simulations

In [ ]:
for case in case_list:
    dfsim2 = dfsim[ dfsim.case == case ]
    print(f"{case:3} has {len(dfsim2)} LFC cutoff simulations")

In [ ]:
want_review_data = False

if want_review_data:
    for case in case_list:
        mtd.open_case(case, verbose=False)
        
        fname, fname_ori, title = mtd.set_lfc_names()
        print(f"fname '{fname}' and title '{title}'")
        print(f"LFC cutoff: lfc={mtd.abs_lfc_cutoff:.3f} fdr={mtd.fdr_lfc_cutoff}")
        
        print("")
        mtd.echo_parameters()

### DEGs/DAPs frequency
### Not Normalized

In [ ]:
#dfsim = pdreadcsv( mtd.cfg.fname_lfc_cutoff, mtd.cfg.root_config)
dfsim = mtd.cfg.open_all_lfc_cutoff()
print(len(dfsim))
dfsim.tail(2)

In [ ]:
i=0
case = case_list[i]
print(">>>", case)
df2 = dfsim[dfsim.case == case].copy()
print(len(df2))
df2.head(2)

In [ ]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))
dfsim.head(2)

In [ ]:
i=0
df2 = dfsim[dfsim.case == case_list[i]]
len(df2)

In [ ]:
fdr = 0.05
mtd.LFC_cut_inf, fdr

In [ ]:
df2 = df2.sort_values(['fdr_lfc_cutoff', 'abs_lfc_cutoff'], ascending=[True, False])

dfsim2 = df2[ (df2.fdr_lfc_cutoff == fdr) & (df2.abs_lfc_cutoff >= mtd.LFC_cut_inf) ]

cols2=['n_degs', 'fdr_lfc_cutoff', 'abs_lfc_cutoff']

dfsim2[cols2]

## Calc all Spearman Correlations - filter the 5 best not repeated fdrs
#### Plot abs_LFC x num of DEG/DAPs
#### corr_cutoff = -.90
#### calc corelation with mtd.LFC_cut_inf = 0.40

In [ ]:
want_calc = False
corr_cutoff=-0.90
nregs_fdr = 5

force=False
verbose=False

df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'abs_lfc_cutoff'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, force=force, verbose=verbose)
print(print(len(df_all_fdr)))
print(df_all_fdr.columns)

df_all_fdr[~pd.isnull(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)].head(2)

In [ ]:
df_all_fdr[~pd.isna(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)]['corr'].min()

### Plot all dfsim

In [ ]:
# !pip3 install -U kaleido

In [ ]:
fdr_list

In [ ]:
for case in case_list:
    print(">>>", case)
    dic_fig = mtd.plot_all_dfsim(dfsim, case=case, fdr_list=fdr_list, width=1100, height=700, title=None, verbose=force)
        
    for key, fig in dic_fig.items():
        print("\t", key)
        fig.show()
        break # remove to see Up and Dw

    print("\n")
    

In [ ]:
# dfsim.columnscutoff_list

### Restricting the best fdr by Spearman's Correlation

### Must calc for each LFC_cut_inf

In [ ]:
corr_cutoff=-0.90
nregs_fdr = 10
mtd.LFC_cut_inf = 0

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'abs_lfc_cutoff'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

### Medulloblastoma LFC_cut_inf = 0.65

In [ ]:
dfsim = mtd.dfsim[mtd.dfsim.case == case]
dfsim = dfsim.sort_values(['fdr_lfc_cutoff', 'abs_lfc_cutoff'], ascending=[True, False])

dfsim.fdr_lfc_cutoff.unique(), dfsim.abs_lfc_cutoff.unique()

### For FDR == 0.05 (default cutoff) - there is no correlation, is a horizontal flat line for 0.05

In [ ]:
mtd.LFC_cut_inf

In [ ]:
fdr = 0.05

i=0
case = case_list[i]
print(">>>", case)

dfsim2 = dfsim[ (dfsim.case==case) & (dfsim.fdr_lfc_cutoff == fdr) & (dfsim.abs_lfc_cutoff >= mtd.LFC_cut_inf) ]
len(dfsim2)

In [ ]:
i=1
case = case_list[i]
print(">>>", case)

dfsim2 = dfsim[ (dfsim.case==case) & (dfsim.fdr_lfc_cutoff == fdr) & (dfsim.abs_lfc_cutoff >= mtd.LFC_cut_inf) ]
len(dfsim2)

In [ ]:
cols2=['case', 'n_degs', 'fdr_lfc_cutoff', 'abs_lfc_cutoff']
dfsim2[cols2].head(5)

In [ ]:
dfsim2[cols2].tail(5)

In [ ]:
dfsim3 = dfsim2.reset_index()

cols3=['index', 'n_degs', 'fdr_lfc_cutoff', 'abs_lfc_cutoff']

dfsim3[cols3].head(2)

In [ ]:
dfsim3[cols3].iloc[:, [0,1]].head(3)

In [ ]:
method='spearman'
corr = dfsim3[cols3].iloc[:, [0,1]].corr(method=method)
corr

In [ ]:
pd.isnull(corr)

### mtd.LFC_cut_inf = 0.40

In [ ]:
nregs_fdr = 10

LFC_cut_inf = dic_yml['LFC_cut_inf']
mtd.LFC_cut_inf = LFC_cut_inf

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'abs_lfc_cutoff'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

### WNT - Spearman

In [ ]:
i=0
case = case_list[i]

df2 = df_all_fdr[ (df_all_fdr.case == case) ]
print(len(df2))

df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
print(len(df2))
print(f"{mtd.s_deg_dap} max: {df2.n_degs_max.max()}")
df2.head(2)

### G4 Spearman

In [ ]:
i=1
case = case_list[i]

df2 = df_all_fdr[ (df_all_fdr.case == case) ]
print(len(df2))

df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
print(len(df2))
print(f"{mtd.s_deg_dap} max: {df2.n_degs_max.max()}")
df2.head(2)

In [ ]:
for case in case_list:
    df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
    print(f"case: {case:4} n={len(df2)} max {mtd.s_deg_dap}: {df2.n_degs_max.max()}")

### Plot abs_LFC x num of DEGs/DAPs
  - set LFC_cut_inf

In [ ]:
corr_cutoff, nregs_fdr, case_list

In [ ]:
i=0
case  = case_list[i]
print(">>>", case)

cols2=['n_degs', 'abs_lfc_cutoff']
limit_fdr = -1
method='spearman'

ret, dic_return = mtd.calc_nDEG_curve_per_LFC_FDR(case=case, cols2=cols2, 
                                                  corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                                  method=method, verbose=verbose)

len(dic_return)

In [ ]:
list(dic_return.keys())

In [ ]:
len(dic_return['df_fdr'])

In [ ]:
len(dic_return['name_list']), dic_return['name_list']

In [ ]:
len(dic_return['fdrs']), np.array(dic_return['fdrs'])

In [ ]:
df_fdr = dic_return['df_fdr']
df_fdr.head(2)

In [ ]:
LFC_cut_inf2 = mtd.LFC_cut_inf
corr_cutoff

In [ ]:
mtd.LFC_cut_inf = 0.
verbose = False

i=0
case = case_list[i]
mtd.open_case(case)
print(">>>", case)

ret, dic_fig, df_fdr = mtd.plot_nDEG_curve_per_LFC_FDR(case, width=1100, height=700, title=None, 
                                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=verbose)

for key, fig in dic_fig.items():
    print(key)
    fig.show()

In [ ]:
dic_fig = mtd.plot_all_LFC_FDR_cutoffs(width=1100, height=700, title=None, 
                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=force)

for case in case_list:
    print(">>>", case)
    try:
        dic2 = dic_fig[case]
    except:
        continue
        
    for key, fig in dic2.items():
        print(key)
        fig.show()
        break
    print("")

In [ ]:
mtd.LFC_cut_inf = LFC_cut_inf2
mtd.LFC_cut_inf

## LFC_cut_inf = 0.65 - see the curve

### Ploting only Spearman's limiar curves

In [ ]:
dic_fig = mtd.plot_all_LFC_FDR_cutoffs(width=1100, height=700, title=None, 
                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=force)

for case in case_list:
    print(">>>", case)
    try:
        dic2 = dic_fig[case]
    except:
        continue
        
    for key, fig in dic2.items():
        fig.show()
        # do not show Up and Down
        break
    print("")

In [ ]:
LFC_cut_inf = mtd.LFC_cut_inf
LFC_cut_inf

In [ ]:
case = case_list[0]

df_all_fdr = mtd.open_fdr_lfc_correlation(case, LFC_cut_inf)
df2 = df_all_fdr[ pd.notnull(df_all_fdr['corr']) ]
print(len(df2))
df2.head(6)

In [ ]:
case = case_list[1]

df_all_fdr = mtd.open_fdr_lfc_correlation(case, LFC_cut_inf)
df2 = df_all_fdr[ pd.notnull(df_all_fdr['corr']) ]
print(len(df2))
df2.head(6)

### Summary DEG/DAPs + Up and Down (pre-best cutoff)

In [ ]:
verbose=False
per_biotype= False
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa.T

### per_biotype

In [ ]:
verbose=False
per_biotype= True
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa

### Barplot per per_biotype

In [ ]:
per_biotype = True
ensembl = True
before_best_cutoff = True
fig, dfa = mtd.barplot_up_down_genes_per_case(per_biotype=per_biotype, ensembl=ensembl, before_best_cutoff=before_best_cutoff, width=1100, height=700, verbose=False)
if fig: fig.show()

In [ ]:
width = 1300

fig = mtd.plot_all_degs_up_down_per_cutoffs(width=width, height=600, title='', y_anchor=1.05, color_up='darkred', color_dw='navy', plot_bgcolor='whitesmoke', verbose=False)
if fig: fig.show()

### Simple summary

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=False, ensembl=False, verbose=False)
dfa

### per Biotype

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=False, verbose=False)
dfa

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=True, verbose=False)
dfa

### Review data

In [ ]:
want_review_data = True

if want_review_data:
    i=0
    case = case_list[i]
    mtd.open_case(case, verbose=False)
    
    fname, fname_ori, title = mtd.set_lfc_names()
    print(f"fname '{fname}' and title '{title}'")
    print(f"LFC cutoff: lfc={mtd.abs_lfc_cutoff:.3f} fdr={mtd.fdr_lfc_cutoff}")
    
    print("")
    mtd.echo_parameters()

In [ ]:
want_review_data = True

if want_review_data:

    for case in case_list:
        mtd.open_case(case, verbose=False)
        
        fname, fname_ori, title = mtd.set_lfc_names()
        print(">>>", case)
        print(f"fname '{fname}' and title '{title}'")
        print(f"LFC cutoff: lfc={mtd.abs_lfc_cutoff:.3f} fdr={mtd.fdr_lfc_cutoff}")
        
        print("")
        mtd.echo_parameters()

### LNCs

In [ ]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(len(dflfc_lnc))
dflfc_lnc.tail(3)

In [ ]:
np.unique(dflfc_lnc._type)

In [ ]:
np.unique(dflfc_lnc.biotype)

In [ ]:
cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type', 'lfc', 'abs_lfc', 'pval', 'fdr', 'mean_exp', 't', 'B', 'description', 
        'desc_gff', 'description_prev',   'accession', 'ensembl_id', 'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe',  'seqname', 'start', 'end', 'go_id', 'seq']

cols = ['probe', 'symbol', 'biotype', '_type', 'lfc', 'fdr', 'desc_gff', 'accession', 'ensembl_id', 'ensembl_transc_id', 'cytoband', 'seqname', 'start', 'end', 'seq']
print(len(dflfc_lnc))

dflfc_lnc = dflfc_lnc.sort_values('abs_lfc', ascending=False)
df2 = dflfc_lnc[cols]
df2.head(30)

In [ ]:
fname = 'microarray_ncRNAs.tsv'
pdwritecsv(df2, fname, mtd.root_result, verbose=True)

### Havana is the Ensembl curation project

In [ ]:
dfgff = mtd.gene.prepare_final_gff(force=False, verbose=True)
print(len(dfgff))
dfgff.head(3)

### DEGs/DAPs frequency
### Not Normalized

In [ ]:
#dfsim = pdreadcsv( mtd.cfg.fname_lfc_cutoff, mtd.cfg.root_config)
dfsim = mtd.cfg.open_all_lfc_cutoff()
print(len(dfsim))
dfsim.tail(3)

### WNT

In [ ]:
mtd.set_db(0)

i=0
case = case_list[i]
print(">>>", case)
df2 = dfsim[dfsim.case == case]
print(len(df2))
df2.head(3)

### G4

In [ ]:
i=1
case = case_list[i]
print(">>>", case)
df2 = dfsim[dfsim.case == case]
print(len(df2))
df2.head(3)